In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip -q install mediapipe kagglehub scikit-learn
!wget -q -nc https://storage.googleapis.com/mediapipe-models/face_detector/blaze_face_short_range/float16/1/blaze_face_short_range.tflite -O /content/blaze_face_short_range.tflite
print("Setup complete.")

Mounted at /content/drive
Setup complete.


In [2]:
import os, shutil, json, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet18
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision

P = "/content/drive/MyDrive/Multimedia_Authentication"
MODELS_DIR = f"{P}/v2/models"
CKPTS = {
    "gated":     f"{MODELS_DIR}/gated_v2_best.pth",
    "spatial":   f"{MODELS_DIR}/spatial_resnet18_v2_best.pth",
    "frequency": f"{MODELS_DIR}/frequency_resnet18_v2_best.pth",
}
OUT = f"{P}/v2/results"
FIG = f"{OUT}/figs"
FACE_CACHE = f"{P}/cross_dataset_cache/faces_dfd"
for d in (OUT, FIG, FACE_CACHE):
    os.makedirs(d, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
for name, path in CKPTS.items():
    print(f"{name:10s} checkpoint exists:", os.path.exists(path))
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing checkpoint: {path}. Update CKPTS to the exact file.")
if device.type != "cuda":
    warnings.warn("GPU not detected. This will be slow on CPU.")

Device: cpu
gated      checkpoint exists: True
spatial    checkpoint exists: True
frequency  checkpoint exists: True


/tmp/ipykernel_1448/664206399.py:39: UserWarning: GPU not detected. This will be slow on CPU.
  warnings.warn("GPU not detected. This will be slow on CPU.")


## 1. Face extraction (identical to the training and earlier cross-dataset pipeline)
8 evenly spaced frames, largest detected face, nearest box reused when a frame has no detection, 1.4x square crop resized to 224x224.

In [3]:
def make_detector():
    options = vision.FaceDetectorOptions(
        base_options=mp_python.BaseOptions(model_asset_path="/content/blaze_face_short_range.tflite"),
        min_detection_confidence=0.5
    )
    return vision.FaceDetector.create_from_options(options)


def read_frames(video_path, n=8):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return None
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        return None
    frames = []
    for frame_idx in np.linspace(0, total - 1, n, dtype=int):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
        ok, frame = cap.read()
        if not ok:
            cap.release()
            return None
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()
    return frames


def detect_box(detector, frame, max_side=640):
    height, width = frame.shape[:2]
    scale = min(1.0, max_side / max(height, width))
    small = (cv2.resize(frame, (int(width * scale), int(height * scale)), interpolation=cv2.INTER_AREA)
             if scale < 1.0 else frame)
    result = detector.detect(mp.Image(image_format=mp.ImageFormat.SRGB,
                                      data=np.ascontiguousarray(small, dtype=np.uint8)))
    if not result.detections:
        return None
    box = max(result.detections, key=lambda d: d.bounding_box.width * d.bounding_box.height).bounding_box
    return (box.origin_x / scale, box.origin_y / scale, box.width / scale, box.height / scale)


def square_crop(frame, box, scale=1.4, out=224):
    height, width = frame.shape[:2]
    x, y, w, h = box
    cx, cy = x + w / 2, y + h / 2
    side = int(min(max(w, h) * scale, height, width))
    side = max(side, 2)
    x1 = int(np.clip(cx - side / 2, 0, width - side))
    y1 = int(np.clip(cy - side / 2, 0, height - side))
    crop = frame[y1:y1 + side, x1:x1 + side]
    if crop.size == 0:
        raise ValueError("Empty face crop")
    return cv2.resize(crop, (out, out), interpolation=cv2.INTER_AREA)


def process_video(detector, video_path):
    frames = read_frames(video_path)
    if frames is None:
        return None, "read_failed"
    boxes = [detect_box(detector, frame) for frame in frames]
    found = [i for i, b in enumerate(boxes) if b is not None]
    if not found:
        return None, "no_face"
    for i, b in enumerate(boxes):
        if b is None:
            boxes[i] = boxes[min(found, key=lambda j: abs(j - i))]
    crops = [square_crop(f, b) for f, b in zip(frames, boxes)]
    return np.stack(crops).astype(np.uint8), "ok"

## 2. Load the trained models (strict loading: a key mismatch fails loudly)

In [4]:
MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
STD  = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)


def haar_highfreq(x):
    a = x[..., 0::2, 0::2]; b = x[..., 0::2, 1::2]
    c = x[..., 1::2, 0::2]; d = x[..., 1::2, 1::2]
    lh = (a + b - c - d) / 2
    hl = (a - b + c - d) / 2
    hh = (a - b - c + d) / 2
    return torch.stack([lh.mean(1), hl.mean(1), hh.mean(1)], dim=1)


def to_branch(x, branch):
    # x: [B, T, 3, H, W] in 0..1
    B, T, C, H, W = x.shape
    x = x.reshape(B * T, C, H, W)
    if branch == "spatial":
        x = (x - MEAN) / STD
    else:
        h = haar_highfreq(x)
        mn = h.amin(dim=(1, 2, 3), keepdim=True)
        mx = h.amax(dim=(1, 2, 3), keepdim=True)
        h = (h - mn) / (mx - mn + 1e-8)
        h = F.interpolate(h, size=(H, W), mode="bilinear", align_corners=False)
        x = (h - MEAN) / STD
    return x.reshape(B, T, C, H, W)


def backbone():
    return nn.Sequential(*list(resnet18(weights=None).children())[:-1])


class BranchNet(nn.Module):
    # Standalone single-branch model (same layout as training: backbone.*, classifier.*)
    def __init__(self):
        super().__init__()
        self.backbone = backbone()
        self.classifier = nn.Linear(512, 2)

    def forward(self, x):
        B, T = x.shape[:2]
        f = self.backbone(x.flatten(0, 1)).flatten(1)
        return self.classifier(f.view(B, T, 512).mean(1))


class GatedFusionNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.spatial_bb = backbone()
        self.freq_bb = backbone()
        self.gate = nn.Sequential(nn.Linear(1024, 128), nn.ReLU(), nn.Dropout(0.2), nn.Linear(128, 1))
        self.classifier = nn.Sequential(nn.Linear(512, 256), nn.ReLU(), nn.Dropout(0.5), nn.Linear(256, 2))

    def feats(self, x, bb):
        B, T = x.shape[:2]
        return bb(x.flatten(0, 1)).flatten(1).view(B, T, 512).mean(1)


def load(model, path):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    model.load_state_dict(ck["model_state_dict"], strict=True)
    model.to(device).eval()
    print(f"loaded {Path(path).name} | val AUC in checkpoint: {ck.get('val_auc')}")
    return model


gated = load(GatedFusionNet(), CKPTS["gated"])
spatial_net = load(BranchNet(), CKPTS["spatial"])
freq_net = load(BranchNet(), CKPTS["frequency"])


def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


@torch.no_grad()
def predict_all(frames):
    x = (torch.from_numpy(frames).to(device).unsqueeze(0)
         .permute(0, 1, 4, 2, 3).float().div(255.0))
    with torch.autocast("cuda", dtype=torch.float16, enabled=(device.type == "cuda")):
        xs, xf = to_branch(x, "spatial"), to_branch(x, "frequency")

        # gated model: compute branch features once, then mix
        s = gated.feats(xs, gated.spatial_bb)
        f = gated.feats(xf, gated.freq_bb)
        alpha = torch.sigmoid(gated.gate(torch.cat([s, f], 1)).float())

        def mix(a):
            a = a.to(s.dtype)
            return gated.classifier(a * s + (1 - a) * f)

        lg_gated = mix(alpha)
        lg_a1 = mix(torch.full_like(alpha, 1.0))
        lg_a05 = mix(torch.full_like(alpha, 0.5))
        lg_a0 = mix(torch.full_like(alpha, 0.0))

        # separately trained standalone branches
        lg_s = spatial_net(xs)
        lg_f = freq_net(xf)

    pf = lambda lg: float(torch.softmax(lg.float(), 1)[0, 1])
    p_s, p_f = pf(lg_s), pf(lg_f)
    p_late = float(1 / (1 + np.exp(-(logit(p_s) + logit(p_f)) / 2)))
    return {
        "p_gated": pf(lg_gated), "alpha": float(alpha[0, 0]),
        "p_spatial": p_s, "p_frequency": p_f, "p_late": p_late,
        "p_alpha1": pf(lg_a1), "p_alpha05": pf(lg_a05), "p_alpha0": pf(lg_a0),
    }

loaded gated_v2_best.pth | val AUC in checkpoint: 0.9863362351569898
loaded spatial_resnet18_v2_best.pth | val AUC in checkpoint: 0.9871057040868362
loaded frequency_resnet18_v2_best.pth | val AUC in checkpoint: 0.9649595687331536


## 3. Pick a small balanced sample of DFD (no full download)
The Kaggle DFD copy is about 24 GB (3,068 manipulated + the original sequences). This notebook **never downloads the whole dataset**. It lists the file names, picks `N_PER_CLASS` real and `N_PER_CLASS` fake videos at random, and downloads only those (each is roughly 1 to 9 MB). With the default of 100 per class that is about 1 GB.

One-time setup for the file listing: the notebook needs your Kaggle login only to list file names. The first time you run section 3 it asks for an API token in a hidden box (Kaggle > Settings > API > create a token). You can instead save Colab secrets (key icon in the sidebar) named `KAGGLE_API_TOKEN`, or `KAGGLE_USERNAME` and `KAGGLE_KEY`. The file list is saved on Drive, so this is only needed once. Never paste the token into a text cell or share it.

In [5]:
import kagglehub

KAGGLE_HANDLE = "sanikatiwarekar/deep-fake-detection-dfd-entire-original-dataset"
N_PER_CLASS = 100        # real and fake videos to use; lower it (e.g. 50) for a lighter run
RANDOM_SEED = 42
DFD_LOCAL = f"{P}/cross_dataset_cache/DFD"
os.makedirs(DFD_LOCAL, exist_ok=True)

# Kaggle credentials: Colab secrets if present, otherwise a hidden prompt (nothing is saved to the notebook)
import getpass


def setup_kaggle_auth():
    try:
        from google.colab import userdata
        for key in ("KAGGLE_API_TOKEN", "KAGGLE_USERNAME", "KAGGLE_KEY"):
            try:
                os.environ[key] = userdata.get(key)
            except Exception:
                pass
    except Exception:
        pass

    if os.environ.get("KAGGLE_API_TOKEN") or (os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY")):
        print("Kaggle credentials found.")
        return

    print("No Kaggle credentials in Colab secrets.")
    print("On Kaggle: Settings > API, create an API token (or a legacy API key).")
    token = getpass.getpass("Paste the Kaggle API token (hidden). Press Enter to use username + legacy key instead: ").strip()
    if token:
        os.environ["KAGGLE_API_TOKEN"] = token
    else:
        os.environ["KAGGLE_USERNAME"] = input("Kaggle username: ").strip()
        os.environ["KAGGLE_KEY"] = getpass.getpass("Kaggle legacy API key (hidden): ").strip()


def list_dataset_files(handle):
    from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi()
    try:
        api.authenticate()
    except BaseException as exc:      # the Kaggle client may call exit() on failure
        raise RuntimeError("Kaggle authentication failed. Re-run this cell and paste a valid token "
                           "(Kaggle > Settings > API).") from exc
    names, token = [], None
    while True:
        try:
            res = api.dataset_list_files(handle, page_token=token, page_size=200)
        except TypeError:
            res = api.dataset_list_files(handle)
        files = getattr(res, "files", None) or []
        names += [getattr(f, "name", None) or getattr(f, "ref", None) for f in files]
        token = getattr(res, "next_page_token", None) or getattr(res, "nextPageToken", None)
        if not token:
            break
    return [n for n in names if n]


list_csv = f"{DFD_LOCAL}/dfd_file_list.csv"
if os.path.isfile(list_csv):
    names = pd.read_csv(list_csv)["name"].tolist()
    print("Loaded saved file list:", len(names), "files")
else:
    setup_kaggle_auth()
    names = list_dataset_files(KAGGLE_HANDLE)
    pd.DataFrame({"name": names}).to_csv(list_csv, index=False)
    print("Listed and saved:", len(names), "files")

videos = [n for n in names if n.lower().endswith(".mp4")]


def label_of(path):
    low = path.lower()
    if "manipulated" in low:
        return "FAKE"
    if "original" in low:
        return "REAL"
    return None


manifest = pd.DataFrame({"relative_path": videos, "label": [label_of(v) for v in videos]})
print("Top-level folders found:", sorted({v.split("/")[0] for v in videos}))
unknown = manifest[manifest["label"].isna()]
if len(unknown):
    print(f"Ignoring {len(unknown)} videos whose folder is neither original nor manipulated, e.g.:",
          unknown["relative_path"].head(3).tolist())
manifest = manifest.dropna(subset=["label"]).reset_index(drop=True)
print(manifest["label"].value_counts())
if manifest["label"].nunique() < 2 or len(manifest) < 2 * N_PER_CLASS:
    raise RuntimeError("The file list is incomplete (an old Kaggle API may return only the first page). "
                       "Run: !pip -q install -U kaggle, restart the runtime, delete dfd_file_list.csv on Drive and retry.")

rng_sel = np.random.default_rng(RANDOM_SEED)
parts = []
for label in ["REAL", "FAKE"]:
    g = manifest[manifest["label"] == label]
    idx = rng_sel.choice(g.index.to_numpy(), size=min(N_PER_CLASS, len(g)), replace=False)
    parts.append(g.loc[idx])
selected = pd.concat(parts).sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
print("\nSelected:", len(selected))
print(selected["label"].value_counts())

Loaded saved file list: 3432 files
Top-level folders found: ['DFD_manipulated_sequences', 'DFD_original sequences']
label
FAKE    3068
REAL     363
Name: count, dtype: int64

Selected: 200
label
REAL    100
FAKE    100
Name: count, dtype: int64


## 4. Download only the selected videos (cached on Drive; the temporary copy is deleted to save Colab disk)

In [6]:
import time


def safe_relative_path(rel):
    norm = os.path.normpath(rel)
    if norm.startswith("..") or os.path.isabs(norm):
        raise ValueError(f"Unsafe relative path: {rel}")
    return norm


items, download_errors = [], []
for row in tqdm(selected.itertuples(index=False), total=len(selected), desc="Downloading selected videos"):
    rel = safe_relative_path(row.relative_path)
    local_path = os.path.join(DFD_LOCAL, rel)
    os.makedirs(os.path.dirname(local_path), exist_ok=True)

    if not os.path.isfile(local_path) or os.path.getsize(local_path) == 0:
        try:
            src_path, last_exc = None, None
            for attempt in range(5):          # Colab's download helper sometimes returns a transient 500 error
                try:
                    src_path = str(kagglehub.dataset_download(KAGGLE_HANDLE, path=row.relative_path))
                    break
                except Exception as exc:
                    last_exc = exc
                    time.sleep(3 * (attempt + 1))
            if src_path is None:
                raise last_exc
            if os.path.isdir(src_path):
                matches = list(Path(src_path).rglob(os.path.basename(rel)))
                if len(matches) != 1:
                    raise FileNotFoundError(f"Could not locate {rel} in {src_path}")
                src_path = str(matches[0])
            shutil.copy2(src_path, local_path)
            if os.path.abspath(src_path) != os.path.abspath(local_path):
                try:
                    os.remove(src_path)      # free Colab disk; the copy lives on Drive
                except OSError:
                    pass
        except Exception as exc:
            download_errors.append({"relative_path": rel, "label": row.label, "error": repr(exc)})
            continue

    if os.path.isfile(local_path) and os.path.getsize(local_path) > 0:
        items.append((local_path, rel, row.label))

print(f"\nReady videos: {len(items)} / {len(selected)} | download failures: {len(download_errors)}")
if download_errors:
    pd.DataFrame(download_errors).to_csv(f"{OUT}/dfd_download_errors.csv", index=False)
    print("First error:", download_errors[0]["error"])
if not items:
    raise RuntimeError("No videos downloaded. Check the Kaggle login and file names.")
print(pd.Series([l for _, _, l in items]).value_counts())


100%|██████████| 3.32M/3.32M [00:00<00:00, 39.0MB/s]



  0%|          | 0.00/13.9M [00:00<?, ?B/s]
100%|██████████| 13.9M/13.9M [00:00<00:00, 86.0MB/s]



  0%|          | 0.00/14.5M [00:00<?, ?B/s]
 14%|█▍        | 2.00M/14.5M [00:00<00:00, 19.0MB/s]
 62%|██████▏   | 9.00M/14.5M [00:00<00:00, 47.5MB/s]
100%|██████████| 14.5M/14.5M [00:00<00:00, 45.8MB/s]



100%|██████████| 3.85M/3.85M [00:00<00:00, 43.8MB/s]



  0%|          | 0.00/9.14M [00:00<?, ?B/s]
 55%|█████▍    | 5.00M/9.14M [00:00<00:00, 37.2MB/s]
100%|██████████| 9.14M/9.14M [00:00<00:00, 30.5MB/s]



100%|██████████| 2.02M/2.02M [00:00<00:00, 24.9MB/s]


... (71 outputs left out)


  0%|          | 0.00/7.51M [00:00<?, ?B/s]
100%|██████████| 7.51M/7.51M [00:00<00:00, 67.9MB/s]



  0%|          | 0.00/4.97M [00:00<?, ?B/s]
100%|██████████| 4.97M/4.97M [00:00<00:00, 49.5MB/s]



  0%|          | 0.00/7.16M [00:00<?, ?B/s]
100%|██████████| 7.16M/7.16M [00:00<00:00, 51.5MB/s]



Ready videos: 200 / 200 | download failures: 0
REAL    100
FAKE    100
Name: count, dtype: int64


## 5. Run inference (all models, one pass per video)

In [7]:
detector = make_detector()
rows = []

for local_path, rel, label in tqdm(items, desc="Evaluating"):
    cache_file = os.path.join(FACE_CACHE, rel.replace("/", "__").replace("\\", "__") + ".npy")
    try:
        if os.path.isfile(cache_file):
            faces, status = np.load(cache_file), "ok"
        else:
            faces, status = process_video(detector, local_path)
            if faces is not None:
                np.save(cache_file, faces)
        if faces is None:
            rows.append({"path": rel, "label": label, "status": status})
            continue
        res = predict_all(faces)
        rows.append({"path": rel, "label": label, "status": "ok", **res})
    except Exception as exc:
        rows.append({"path": rel, "label": label, "status": f"error: {type(exc).__name__}: {exc}"})

pred = pd.DataFrame(rows)
csv_path = f"{OUT}/cross_DFD_standalone_branches_predictions.csv"
pred.to_csv(csv_path, index=False)

valid = pred[pred["status"] == "ok"].copy()
print("Saved:", csv_path)
print(f"Evaluated {len(valid)} / {len(pred)} videos | skipped {len(pred) - len(valid)}")
print(pred.loc[pred.status != "ok", ["path", "label", "status"]].to_string(index=False))
print(valid["label"].value_counts())

Evaluating:   0%|          | 0/200 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/Multimedia_Authentication/v2/results/cross_DFD_standalone_branches_predictions.csv
Evaluated 174 / 200 videos | skipped 26
                                                                                                   path label  status
                                            DFD_original sequences/23__outside_talking_pan_laughing.mp4  REAL no_face
DFD_manipulated_sequences/DFD_manipulated_sequences/18_06__outside_talking_still_laughing__DEA1TCLN.mp4  FAKE no_face
               DFD_manipulated_sequences/DFD_manipulated_sequences/01_11__meeting_serious__FAFWDR4W.mp4  FAKE no_face
                                                    DFD_original sequences/20__walk_down_hall_angry.mp4  REAL no_face
  DFD_manipulated_sequences/DFD_manipulated_sequences/06_11__outside_talking_pan_laughing__MX659QU8.mp4  FAKE no_face
DFD_manipulated_sequences/DFD_manipulated_sequences/13_03__outside_talking_still_laughing__GBYWJW06.mp4  FAKE no_face
                     

## 6. Results at the fixed threshold 0.5
Rows marked "forced alpha" use the gated model's classifier with the mixing weight fixed by hand, so they show how much the learned gate adds. AUC is threshold-free. Do not choose a threshold from this table.

In [8]:
y = (valid["label"].values == "FAKE").astype(int)
real_idx, fake_idx = np.flatnonzero(y == 0), np.flatnonzero(y == 1)
rng = np.random.default_rng(42)

MODELS = {
    "Gated fusion (learned alpha)":  "p_gated",
    "Spatial branch alone":          "p_spatial",
    "Frequency branch alone":        "p_frequency",
    "Late fusion (mean of logits)":  "p_late",
    "Gated, alpha forced 1.0":       "p_alpha1",
    "Gated, alpha forced 0.5":       "p_alpha05",
    "Gated, alpha forced 0.0":       "p_alpha0",
}
B = 1000
boot_idx = [np.concatenate([rng.choice(real_idx, len(real_idx), replace=True),
                            rng.choice(fake_idx, len(fake_idx), replace=True)]) for _ in range(B)]


def boot_ci(p):
    aucs = [roc_auc_score(y[i], p[i]) for i in boot_idx]
    return np.percentile(aucs, [2.5, 97.5])


rows = []
for name, col in MODELS.items():
    p = valid[col].to_numpy()
    pred_lab = (p >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred_lab, labels=[0, 1]).ravel()
    real_recall, fake_recall = tn / (tn + fp), tp / (tp + fn)
    lo, hi = boot_ci(p)
    rows.append({
        "model": name,
        "AUC": round(roc_auc_score(y, p), 4),
        "AUC 95% CI": f"{lo:.3f}-{hi:.3f}",
        "accuracy": round((pred_lab == y).mean(), 4),
        "REAL recall": round(real_recall, 4),
        "FAKE recall": round(fake_recall, 4),
        "balanced acc": round((real_recall + fake_recall) / 2, 4),
        "pred FAKE %": round(100 * pred_lab.mean(), 1),
    })

results = pd.DataFrame(rows)
print(f"DFD sample | REAL {len(real_idx)} | FAKE {len(fake_idx)} | threshold 0.5\n")
print(results.to_string(index=False))
results.to_csv(f"{OUT}/cross_DFD_standalone_branches_results.csv", index=False)

# Paired bootstrap: is a model really different from the gated model in AUC?
print("\nPaired AUC differences vs. gated model (95% CI; 'YES' only if the interval excludes 0):")
pg = valid["p_gated"].to_numpy()
for name, col in MODELS.items():
    if col == "p_gated":
        continue
    pm = valid[col].to_numpy()
    d = [roc_auc_score(y[i], pm[i]) - roc_auc_score(y[i], pg[i]) for i in boot_idx]
    lo, hi = np.percentile(d, [2.5, 97.5])
    print(f"{name:32s} mean {np.mean(d):+.4f}  CI [{lo:+.4f}, {hi:+.4f}]  significant: {'YES' if (lo > 0 or hi < 0) else 'no'}")

print("\nMean alpha  REAL %.3f | FAKE %.3f" % (
    valid.loc[y == 0, "alpha"].mean(), valid.loc[y == 1, "alpha"].mean()))

DFD sample | REAL 84 | FAKE 90 | threshold 0.5

                       model    AUC  AUC 95% CI  accuracy  REAL recall  FAKE recall  balanced acc  pred FAKE %
Gated fusion (learned alpha) 0.7626 0.682-0.830    0.6264       0.2976       0.9333        0.6155         82.2
        Spatial branch alone 0.7520 0.676-0.820    0.6667       0.4167       0.9000        0.6583         74.7
      Frequency branch alone 0.8025 0.734-0.865    0.7126       0.4762       0.9333        0.7048         73.6
Late fusion (mean of logits) 0.8025 0.732-0.865    0.7241       0.5000       0.9333        0.7167         72.4
     Gated, alpha forced 1.0 0.6778 0.594-0.755    0.6149       0.3690       0.8444        0.6067         74.1
     Gated, alpha forced 0.5 0.7754 0.697-0.841    0.6437       0.3333       0.9333        0.6333         80.5
     Gated, alpha forced 0.0 0.8106 0.745-0.870    0.6667       0.4048       0.9111        0.6579         75.9

Paired AUC differences vs. gated model (95% CI; 'YES' only if t

## 7. Figures for the report

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
for name in ["Gated fusion (learned alpha)", "Spatial branch alone", "Frequency branch alone", "Late fusion (mean of logits)"]:
    fpr, tpr, _ = roc_curve(y, valid[MODELS[name]].to_numpy())
    ax.plot(fpr, tpr, label=f"{name} (AUC {roc_auc_score(y, valid[MODELS[name]]):.3f})")
ax.plot([0, 1], [0, 1], "k--", lw=1, label="Chance")
ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
ax.set_title("DFD (trained on FF++ C23)")
ax.legend(fontsize=8, loc="lower right")
fig.tight_layout(); fig.savefig(f"{FIG}/dfd_roc_standalone.png", dpi=200); plt.show()

fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
for ax, (title, col) in zip(axes, [("Gated: P(fake)", "p_gated"), ("Spatial branch: P(fake)", "p_spatial"),
                                   ("Frequency branch: P(fake)", "p_frequency")]):
    ax.hist(valid.loc[y == 0, col], bins=25, alpha=0.6, label="REAL", range=(0, 1))
    ax.hist(valid.loc[y == 1, col], bins=25, alpha=0.6, label="FAKE", range=(0, 1))
    ax.set_title(title); ax.set_xlabel("probability"); ax.legend()
fig.tight_layout(); fig.savefig(f"{FIG}/dfd_prob_distributions.png", dpi=200); plt.show()

fig, ax = plt.subplots(figsize=(5.5, 3.8))
ax.hist(valid.loc[y == 0, "alpha"], bins=25, alpha=0.6, label="REAL")
ax.hist(valid.loc[y == 1, "alpha"], bins=25, alpha=0.6, label="FAKE")
ax.set_xlabel("alpha (spatial weight)"); ax.set_title("Gate weight on DFD"); ax.legend()
fig.tight_layout(); fig.savefig(f"{FIG}/dfd_alpha.png", dpi=200); plt.show()
print("Figures saved to:", FIG)

## 8. Save a summary

In [9]:
summary = {
    "dataset": "DFD (Kaggle: sanikatiwarekar/deep-fake-detection-dfd-entire-original-dataset)",
    "split": "random balanced sample, seed 42",
    "n_per_class_setting": N_PER_CLASS,
    "videos_evaluated": int(len(valid)),
    "real": int((y == 0).sum()), "fake": int((y == 1).sum()),
    "evaluation_only": True, "fine_tuning": False, "threshold": 0.5,
    "checkpoints": CKPTS,
}
with open(f"{OUT}/cross_DFD_standalone_branches_summary.json", "w") as fh:
    json.dump(summary, fh, indent=2)
print(json.dumps(summary, indent=2))

{
  "dataset": "DFD (Kaggle: sanikatiwarekar/deep-fake-detection-dfd-entire-original-dataset)",
  "split": "random balanced sample, seed 42",
  "n_per_class_setting": 100,
  "videos_evaluated": 174,
  "real": 84,
  "fake": 90,
  "evaluation_only": true,
  "fine_tuning": false,
  "threshold": 0.5,
  "checkpoints": {
    "gated": "/content/drive/MyDrive/Multimedia_Authentication/v2/models/gated_v2_best.pth",
    "spatial": "/content/drive/MyDrive/Multimedia_Authentication/v2/models/spatial_resnet18_v2_best.pth",
    "frequency": "/content/drive/MyDrive/Multimedia_Authentication/v2/models/frequency_resnet18_v2_best.pth"
  }
}
